# 01 — Dédup des sessions de sommeil qui se chevauchent

Samsung enregistre parfois la même nuit deux fois (ou une sieste à cheval sur une nuit).
Avant tout calcul circadien, il faut décider **quelles sessions ne font qu'un seul épisode de sommeil**.

Deux règles, comme darkhour :

| Usage | Règle de fusion | Pourquoi |
|---|---|---|
| **Analyse** (τ, périodogramme…) | dès que deux sessions se chevauchent, même d'une minute | un épisode ne doit jamais compter deux fois dans le rythme |
| **Affichage** (liste des nuits) | chevauchement ≥ 80 % de la plus courte | on ne cache que les vrais doublons, une sieste collée reste visible |

**Écart volontaire de Nightfall** (section 5) : pour l'analyse, on fusionne aussi les sessions **bout à bout** (écart nul).
C'est la signature des corrections manuelles : quand la montre s'arrête (batterie), la session ajoutée à la main démarre pile à la fin de celle de la montre.

**Critères de validation** (les `assert` plus bas) :
1. Cas synthétiques à vérité connue : chaque règle donne exactement le résultat attendu.
2. Export réel du 24/09 : 6 paires qui se chevauchent, dont 5 à ≥ 80 %.
3. **Parité darkhour** : sur les données de son relevé du 23/09, on retrouve **1146** épisodes.
4. **Règle Nightfall** : 14 contacts bout à bout → **1134** épisodes d'analyse.

# 1. Paramètres

In [ ]:
import pandas as pd
import plotly.express as px

from helpers import charger_export, date_fr, vers_heure_locale

CHEMIN_EXPORT = "/data2/Nightfall/nightfall-export-2026-09-24.zip"

# Seuil « doublon » pour l'affichage : part du chevauchement dans la session la plus courte
SEUIL_DOUBLON = 0.8

# Relevé darkhour fait le 23/09 : il ne voyait pas encore la session commencée ce jour-là
RELEVE_DARKHOUR = pd.Timestamp("2026-09-23 00:00", tz="Europe/Paris")
EPISODES_DARKHOUR = 1146

# 2. Les briques du calcul

## 2.1 Chevauchement entre deux sessions

Deux intervalles `[début, fin[` se chevauchent sur `min(fins) − max(débuts)`.
Si le résultat est négatif ou nul, ils ne se touchent pas (ou se touchent bout à bout, ce qui n'est **pas** un chevauchement).

In [ ]:
def duree_chevauchement(debut_a, fin_a, debut_b, fin_b):
    # Prend deux intervalles et retourne la durée commune (timedelta), zéro s'ils sont disjoints
    commun = min(fin_a, fin_b) - max(debut_a, debut_b)
    return max(commun, pd.Timedelta(0))

## 2.2 Trouver toutes les paires qui se chevauchent

Les sessions sont triées par début. Pour chaque session, on regarde les suivantes
**tant qu'elles commencent avant sa fin** : au-delà, plus aucune ne peut la chevaucher (elles commencent encore plus tard).

In [ ]:
def paires_qui_se_chevauchent(sessions):
    # Prend un DataFrame de sessions (id, start_utc, end_utc)
    # et retourne un DataFrame avec une ligne par paire qui se chevauche :
    #   id_a, id_b, chevauchement, part_de_la_plus_courte (entre 0 et 1)
    sessions = sessions.sort_values("start_utc").reset_index(drop=True)
    paires = []

    for i in range(len(sessions)):
        a = sessions.loc[i]
        for j in range(i + 1, len(sessions)):
            b = sessions.loc[j]
            if b["start_utc"] >= a["end_utc"]:
                break

            chevauchement = duree_chevauchement(a["start_utc"], a["end_utc"], b["start_utc"], b["end_utc"])
            if chevauchement == pd.Timedelta(0):
                continue

            plus_courte = min(a["end_utc"] - a["start_utc"], b["end_utc"] - b["start_utc"])
            paires.append({
                "id_a": a["id"],
                "id_b": b["id"],
                "chevauchement": chevauchement,
                "part_de_la_plus_courte": chevauchement / plus_courte,
            })

    return pd.DataFrame(paires, columns=["id_a", "id_b", "chevauchement", "part_de_la_plus_courte"])

## 2.3 Regrouper les sessions liées

Si A chevauche B et B chevauche C, les trois forment **un seul groupe**, même si A et C ne se touchent pas.
Chaque session démarre seule dans son groupe ; à chaque paire, on verse le groupe de B dans celui de A.

In [ ]:
def regrouper(ids, paires):
    # Prend la liste des ids et les paires (id_a, id_b) à réunir,
    # et retourne un dictionnaire id → identifiant de groupe
    groupe = {id_session: id_session for id_session in ids}

    for id_a, id_b in paires:
        groupe_a, groupe_b = groupe[id_a], groupe[id_b]
        if groupe_a == groupe_b:
            continue
        for id_session, g in groupe.items():
            if g == groupe_b:
                groupe[id_session] = groupe_a

    return groupe

## 2.4 Construire les deux vues

**Épisodes d'analyse** : un groupe devient un seul intervalle, du plus petit début à la plus grande fin.

**Sessions affichées** : dans chaque groupe de doublons, on garde **une** session réelle, la plus riche :
1. le plus de temps couvert par des stades de sommeil,
2. puis la plus longue,
3. puis la plus récemment modifiée.

In [ ]:
def episodes_d_analyse(sessions, paires):
    # Toutes les paires qui se chevauchent sont réunies, quel que soit le taux
    groupe = regrouper(sessions["id"], zip(paires["id_a"], paires["id_b"]))
    avec_groupe = sessions.assign(groupe=sessions["id"].map(groupe))

    episodes = avec_groupe.groupby("groupe").agg(
        start_utc=("start_utc", "min"),
        end_utc=("end_utc", "max"),
        nb_sessions=("id", "count"),
        membres=("id", lambda ids: ";".join(sorted(ids))),
    )
    return episodes.sort_values("start_utc").reset_index()


def sessions_affichees(sessions, paires, couverture_stades, seuil=SEUIL_DOUBLON):
    # Seules les paires au-dessus du seuil sont des doublons
    doublons = paires[paires["part_de_la_plus_courte"] >= seuil]
    groupe = regrouper(sessions["id"], zip(doublons["id_a"], doublons["id_b"]))

    candidates = sessions.assign(
        groupe=sessions["id"].map(groupe),
        couverture_stades=sessions["id"].map(couverture_stades).fillna(pd.Timedelta(0)),
        duree=sessions["end_utc"] - sessions["start_utc"],
    )
    # On trie du meilleur au moins bon, puis on garde la première de chaque groupe
    candidates = candidates.sort_values(
        ["couverture_stades", "duree", "last_modified_utc"],
        ascending=False,
    )
    gardees = candidates.drop_duplicates("groupe", keep="first")
    return gardees.sort_values("start_utc").reset_index(drop=True)

# 3. Validation sur des cas synthétiques (vérité connue)

Chaque cas isole une situation :

| Cas | Sessions (UTC) | Analyse | Affichage |
|---|---|---|---|
| Nuit enregistrée deux fois | A 22:00→06:00, B 23:00→05:00 (dans A) | 1 épisode | 1 session (A, la plus longue) |
| Sieste collée | C 13:00→14:00, D 13:45→16:00 (25 % de C) | 1 épisode | 2 sessions |
| Bout à bout | E 20:00→21:00, F 21:00→22:00 | 2 épisodes | 2 sessions |
| Chaîne | G 00:00→02:00, H 01:30→04:00, I 03:30→05:00 | 1 épisode (G et I ne se touchent pas) | 3 sessions |

In [ ]:
def session_test(id_session, debut, fin):
    return {
        "id": id_session,
        "start_utc": pd.Timestamp(debut, tz="UTC"),
        "end_utc": pd.Timestamp(fin, tz="UTC"),
        "last_modified_utc": pd.Timestamp(fin, tz="UTC"),
    }

synthetique = pd.DataFrame([
    session_test("A", "2026-01-01 22:00", "2026-01-02 06:00"),
    session_test("B", "2026-01-01 23:00", "2026-01-02 05:00"),
    session_test("C", "2026-01-03 13:00", "2026-01-03 14:00"),
    session_test("D", "2026-01-03 13:45", "2026-01-03 16:00"),
    session_test("E", "2026-01-04 20:00", "2026-01-04 21:00"),
    session_test("F", "2026-01-04 21:00", "2026-01-04 22:00"),
    session_test("G", "2026-01-05 00:00", "2026-01-05 02:00"),
    session_test("H", "2026-01-05 01:30", "2026-01-05 04:00"),
    session_test("I", "2026-01-05 03:30", "2026-01-05 05:00"),
])

paires_synth = paires_qui_se_chevauchent(synthetique)
episodes_synth = episodes_d_analyse(synthetique, paires_synth)
affichees_synth = sessions_affichees(synthetique, paires_synth, couverture_stades={})

paires_synth

In [ ]:
# Paires : A-B, C-D, G-H, H-I (E-F bout à bout ne compte pas)
assert set(zip(paires_synth["id_a"], paires_synth["id_b"])) == {("A", "B"), ("C", "D"), ("G", "H"), ("H", "I")}
assert paires_synth.set_index(["id_a", "id_b"]).loc[("A", "B"), "part_de_la_plus_courte"] == 1.0
assert paires_synth.set_index(["id_a", "id_b"]).loc[("C", "D"), "part_de_la_plus_courte"] == 0.25

# Analyse : {A,B} {C,D} {E} {F} {G,H,I}
assert len(episodes_synth) == 5
assert sorted(episodes_synth["nb_sessions"]) == [1, 1, 2, 2, 3]
chaine = episodes_synth[episodes_synth["nb_sessions"] == 3].iloc[0]
assert (chaine["start_utc"], chaine["end_utc"]) == (pd.Timestamp("2026-01-05 00:00", tz="UTC"), pd.Timestamp("2026-01-05 05:00", tz="UTC"))

# Affichage : seul le doublon A/B est fusionné, et c'est A (la plus longue) qui reste
assert list(affichees_synth["id"]) == ["A", "C", "D", "E", "F", "G", "H", "I"]

print("Cas synthétiques : OK")

# 4. Sur l'export réel

## 4.1 Chargement

In [ ]:
export = charger_export(CHEMIN_EXPORT)
sessions = export["sessions"]
stades = export["stades"]

print(f"{len(sessions)} sessions, du {sessions['start_utc'].min():%d/%m/%Y} au {sessions['start_utc'].max():%d/%m/%Y}")
sessions["recording_method"].value_counts()

Temps couvert par des stades pour chaque session : c'est le premier critère pour choisir quelle session garder dans un doublon.

In [ ]:
couverture_stades = (
    stades.assign(duree=stades["end_utc"] - stades["start_utc"])
    .groupby("session_id")["duree"]
    .sum()
)

## 4.2 Les paires qui se chevauchent

In [ ]:
paires = paires_qui_se_chevauchent(sessions)

debuts = sessions.set_index("id")["start_utc"]
decalages = sessions.set_index("id")["start_offset"]
paires["nuit_locale"] = [date_fr(vers_heure_locale(debuts[id_a], decalages[id_a])) for id_a in paires["id_a"]]
paires["chevauchement_min"] = paires["chevauchement"].dt.total_seconds() / 60

paires[["nuit_locale", "chevauchement_min", "part_de_la_plus_courte"]].round(2)

In [ ]:
assert len(paires) == 6, f"{len(paires)} paires au lieu de 6"
assert (paires["part_de_la_plus_courte"] >= SEUIL_DOUBLON).sum() == 5

## 4.3 Les deux vues

In [ ]:
episodes = episodes_d_analyse(sessions, paires)
affichees = sessions_affichees(sessions, paires, couverture_stades)

print(f"Sessions brutes      : {len(sessions)}")
print(f"Épisodes d'analyse   : {len(episodes)}   (tout chevauchement fusionné)")
print(f"Sessions affichées   : {len(affichees)}   (doublons ≥ {SEUIL_DOUBLON:.0%} fusionnés)")
print(f"Taille des groupes   : {episodes['nb_sessions'].value_counts().sort_index().to_dict()}")

In [ ]:
# 4 groupes de 2 sessions, et 1 chaîne de 3 (nuit du 26/09/2025, voir ci-dessous)
assert episodes["nb_sessions"].value_counts().sort_index().to_dict() == {1: 1142, 2: 4, 3: 1}
assert len(episodes) == len(sessions) - 6 == 1147
assert len(affichees) == len(sessions) - 5 == 1148

**La chaîne du 26/09/2025** : un doublon parfait (100 %), puis la session suivante qui commence 9 minutes avant la fin (1,6 %).
L'analyse réunit les trois en un seul épisode ; l'affichage ne fusionne que le doublon et garde la session suivante visible.
C'est le cas « chaîne » des tests synthétiques, présent pour de vrai dans les données.
À noter : la session gardée pour le doublon est la plus courte des deux, car c'est elle qui porte les stades (critère n°1).

In [ ]:
chaine = episodes[episodes["nb_sessions"] == 3].iloc[0]
membres = sessions[(sessions["start_utc"] < chaine["end_utc"]) & (sessions["end_utc"] > chaine["start_utc"])]
membres.assign(
    debut_local=[vers_heure_locale(d, o) for d, o in zip(membres["start_utc"], membres["start_offset"])],
    fin_local=[vers_heure_locale(f, o) for f, o in zip(membres["end_utc"], membres["end_offset"])],
    affichee=membres["id"].isin(affichees["id"]),
)[["id", "debut_local", "fin_local", "affichee"]]

## 4.4 Parité darkhour

darkhour affichait **1146** au relevé du 23/09. On rejoue le calcul en ne gardant que les sessions qu'il pouvait voir.

In [ ]:
vues_par_darkhour = sessions[sessions["start_utc"] < RELEVE_DARKHOUR].reset_index(drop=True)
paires_darkhour = paires_qui_se_chevauchent(vues_par_darkhour)
episodes_darkhour = episodes_d_analyse(vues_par_darkhour, paires_darkhour)
affichees_darkhour = sessions_affichees(vues_par_darkhour, paires_darkhour, couverture_stades)

print(f"Sessions visibles le 23/09 : {len(vues_par_darkhour)}")
print(f"Épisodes d'analyse         : {len(episodes_darkhour)}")
print(f"Sessions affichées         : {len(affichees_darkhour)}")

assert len(episodes_darkhour) == EPISODES_DARKHOUR, f"{len(episodes_darkhour)} au lieu de {EPISODES_DARKHOUR}"
print("Parité darkhour : OK")

Le chiffre de darkhour correspond aux **épisodes d'analyse** (tout chevauchement fusionné), pas aux sessions affichées (1147).
C'est une déduction à partir des comptes : à confirmer en regardant où darkhour affiche ce nombre.

# 5. Règle Nightfall : fusionner aussi les sessions bout à bout

darkhour ne réunit que les sessions qui **se chevauchent**. Or une correction manuelle démarre exactement à la fin de la session de la montre :
elle la **touche** sans la chevaucher, et reste donc un épisode à part. Résultat : la nuit réelle est coupée en deux dans l'analyse.

Sur l'export, 12 des 13 sessions bout à bout sont exactement ce cas : un morceau avec stades (montre) collé à un morceau sans stades (ajout manuel).

On ne fusionne que l'écart **nul**, sans seuil arbitraire : un vrai réveil, même court, laisse un trou et reste séparé.

In [ ]:
def paires_bout_a_bout(sessions):
    # Prend un DataFrame de sessions et retourne les paires (id_a, id_b) où b commence exactement à la fin de a
    sessions = sessions.sort_values("start_utc").reset_index(drop=True)
    paires = []

    for i in range(len(sessions)):
        a = sessions.loc[i]
        for j in range(i + 1, len(sessions)):
            b = sessions.loc[j]
            if b["start_utc"] > a["end_utc"]:
                break
            if b["start_utc"] == a["end_utc"]:
                paires.append({"id_a": a["id"], "id_b": b["id"]})

    return pd.DataFrame(paires, columns=["id_a", "id_b"])


def episodes_nightfall(sessions, paires_chevauchement):
    # Épisodes d'analyse de Nightfall : chevauchements + contacts bout à bout
    contacts = paires_bout_a_bout(sessions)
    return episodes_d_analyse(sessions, pd.concat([paires_chevauchement, contacts], ignore_index=True))

In [ ]:
# Cas synthétiques : seul E-F (20:00→21:00 puis 21:00→22:00) change, et devient un seul épisode
contacts_synth = paires_bout_a_bout(synthetique)
assert set(zip(contacts_synth["id_a"], contacts_synth["id_b"])) == {("E", "F")}
episodes_nf_synth = episodes_nightfall(synthetique, paires_synth)
assert len(episodes_nf_synth) == 4
assert sorted(episodes_nf_synth["nb_sessions"]) == [2, 2, 2, 3]
print("Règle Nightfall, cas synthétiques : OK")

In [ ]:
contacts = paires_bout_a_bout(sessions)
episodes_nf = episodes_nightfall(sessions, paires)

print(f"Contacts bout à bout      : {len(contacts)}")
print(f"Épisodes (règle darkhour) : {len(episodes)}")
print(f"Épisodes (règle Nightfall): {len(episodes_nf)}")
print(f"Taille des groupes        : {episodes_nf['nb_sessions'].value_counts().sort_index().to_dict()}")

assert len(contacts) == 14
assert len(episodes_nf) == 1134

Un des 14 contacts relie deux sessions déjà réunies par un chevauchement (le 26/09/2025) : 13 épisodes de moins, pas 14.

Les épisodes les plus longs, à relire : ce sont eux qui pèseront le plus sur τ si une fusion est abusive.

In [ ]:
episodes_nf["duree_h"] = (episodes_nf["end_utc"] - episodes_nf["start_utc"]).dt.total_seconds() / 3600
decalage_debut = sessions.set_index("id")["start_offset"]

plus_longs = episodes_nf.sort_values("duree_h", ascending=False).head(10)
plus_longs.assign(
    debut_local=[date_fr(vers_heure_locale(d, decalage_debut[g])) for d, g in zip(plus_longs["start_utc"], plus_longs["groupe"])],
    fin_local=[date_fr(vers_heure_locale(f, decalage_debut[g])) for f, g in zip(plus_longs["end_utc"], plus_longs["groupe"])],
)[["debut_local", "fin_local", "duree_h", "nb_sessions"]].round(1)

Sur l'export du 24/09, un seul des 10 plus longs épisodes vient d'une fusion (le 26/09/2025, 4 sessions) :
les autres sont des sessions uniques de 17 à 21 h, déjà longues telles quelles. La règle ne fabrique pas d'épisodes aberrants.

Répartition des durées d'épisode, règle darkhour contre règle Nightfall : la fusion bout à bout ne touche que la queue des longs épisodes.

In [ ]:
comparaison = pd.concat([
    episodes.assign(regle="darkhour", duree_h=(episodes["end_utc"] - episodes["start_utc"]).dt.total_seconds() / 3600),
    episodes_nf.assign(regle="Nightfall"),
])
fig = px.histogram(
    comparaison,
    x="duree_h",
    color="regle",
    barmode="overlay",
    nbins=48,
    opacity=0.6,
    color_discrete_map={"darkhour": "#d37c04", "Nightfall": "#0e9eb0"},
    labels={"duree_h": "Durée de l'épisode (h)", "regle": ""},
)
fig.update_yaxes(title="Nombre d'épisodes", type="log")
fig.update_layout(height=400, hovermode="x unified")
fig.show()

# 6. Visualisation des épisodes fusionnés

Un graphique par épisode d'analyse de plusieurs sessions (règle Nightfall), en heure locale.
En gris l'épisode fusionné (analyse), en bleu les sessions gardées à l'affichage, en orange les doublons masqués.

In [ ]:
multi = episodes_nf[episodes_nf["nb_sessions"] > 1].reset_index(drop=True)
lignes = []

for numero, episode in multi.iterrows():
    membres = sessions[(sessions["start_utc"] < episode["end_utc"]) & (sessions["end_utc"] > episode["start_utc"])]
    premier = membres.iloc[0]
    titre = f"Épisode {numero + 1} — {date_fr(vers_heure_locale(premier['start_utc'], premier['start_offset']))} · {len(membres)} sessions"

    # L'épisode d'analyse en premier : il apparaît ainsi en haut de chaque graphique
    debut = vers_heure_locale(episode["start_utc"], premier["start_offset"])
    fin = vers_heure_locale(episode["end_utc"], premier["start_offset"])
    lignes.append({
        "episode": titre,
        "ligne": "Épisode d'analyse",
        "role": "Épisode d'analyse",
        "debut": debut.tz_localize(None),
        "fin": fin.tz_localize(None),
    })

    for _, s in membres.iterrows():
        debut = vers_heure_locale(s["start_utc"], s["start_offset"])
        fin = vers_heure_locale(s["end_utc"], s["end_offset"])
        lignes.append({
            "episode": titre,
            "ligne": f"{debut:%H:%M} → {fin:%H:%M}",
            "role": "Affichée" if s["id"] in set(affichees["id"]) else "Masquée (doublon)",
            "debut": debut.tz_localize(None),
            "fin": fin.tz_localize(None),
        })

frise = pd.DataFrame(lignes)
# Espace entre deux graphiques : environ 60 px quelle que soit la quantité d'épisodes
ESPACEMENT = 0.3 / len(multi)

fig = px.timeline(
    frise,
    x_start="debut",
    x_end="fin",
    y="ligne",
    color="role",
    facet_row="episode",
    facet_row_spacing=ESPACEMENT,
    color_discrete_map={"Épisode d'analyse": "lightgray", "Affichée": "#0e9eb0", "Masquée (doublon)": "#d37c04"},
    hover_data={"debut": "|%d/%m %H:%M", "fin": "|%d/%m %H:%M", "ligne": False},
)
fig.update_yaxes(title=None, matches=None, autorange="reversed")
fig.update_xaxes(matches=None, showticklabels=True, tickformat="%H:%M")

# Titre de chaque épisode au-dessus de son graphique (et non au milieu, sur les barres)
hauteur = (1 - ESPACEMENT * (len(multi) - 1)) / len(multi)
fig.for_each_annotation(lambda a: a.update(
    text=a.text.replace("episode=", ""), textangle=0,
    x=0, xanchor="left", y=a.y + hauteur / 2, yanchor="bottom",
))
fig.update_layout(height=200 * len(multi), hovermode="closest", legend_title_text=None)
fig.show()

# 7. Conclusion

- Les règles de fusion sont validées sur des cas à vérité connue.
- **Règle darkhour** : 6 paires, 5 vrais doublons, 1147 épisodes d'analyse et 1148 sessions affichées ; **parité atteinte** (1146 sur les données de son relevé).
- **Règle Nightfall** (retenue pour l'analyse) : + 14 contacts bout à bout, **1134 épisodes**. Les corrections manuelles après une panne de montre ne coupent plus une nuit en deux.
- L'affichage reste sur la règle darkhour (doublons ≥ 80 %) : la vue par nuit sera traitée en Phase 4.

Les *golden fixtures* (section 8) figent ces règles pour les tests du `core/` Kotlin.

# 8. Golden fixtures pour le `core/` Kotlin

Le port Kotlin doit donner **exactement** les mêmes résultats que ce notebook. On fige donc des cas d'entrée et leurs sorties attendues,
calculées ici par les fonctions validées plus haut, dans `core/src/test/resources/fixtures/dedup/`.

- **Uniquement des cas inventés** (datés de 2030) : aucune donnée réelle n'entre dans git (C1).
- **Au format du contrat d'export** (`sleep_sessions.csv`, `sleep_stages.csv`) : le test Kotlin les relit avec le lecteur CSV du `core/`.
- Chaque cas isole une situation rencontrée dans les vraies données.

In [ ]:
from pathlib import Path

DOSSIER_FIXTURES = Path("../core/src/test/resources/fixtures/dedup")


def session_fixture(id_session, debut, fin, heures_de_stades=0.0, modifiee=None):
    # Une session inventée ; ses stades (LIGHT) couvrent `heures_de_stades` à partir du début
    return {
        "id": id_session,
        "start_utc": pd.Timestamp(debut, tz="UTC"),
        "end_utc": pd.Timestamp(fin, tz="UTC"),
        "last_modified_utc": pd.Timestamp(modifiee or fin, tz="UTC"),
        "heures_de_stades": heures_de_stades,
    }


CAS_FIXTURES = {
    "session_seule": [
        session_fixture("S1", "2030-01-01 23:00", "2030-01-02 07:00", 8),
    ],
    "doublon_garde_les_stades": [
        session_fixture("LONGUE", "2030-01-01 22:00", "2030-01-02 07:00"),
        session_fixture("MONTRE", "2030-01-01 23:00", "2030-01-02 06:30", 7.5),
    ],
    "doublon_sans_stades_garde_la_plus_longue": [
        session_fixture("COURTE", "2030-01-01 23:00", "2030-01-02 06:00"),
        session_fixture("LONGUE", "2030-01-01 22:30", "2030-01-02 06:30"),
    ],
    "doublon_identique_garde_la_plus_recente": [
        session_fixture("ANCIENNE", "2030-01-01 23:00", "2030-01-02 07:00", modifiee="2030-01-02 08:00"),
        session_fixture("RECENTE", "2030-01-01 23:00", "2030-01-02 07:00", modifiee="2030-01-03 08:00"),
    ],
    "sieste_collee": [
        session_fixture("SIESTE", "2030-01-01 13:00", "2030-01-01 14:00", 1),
        session_fixture("APRES", "2030-01-01 13:45", "2030-01-01 16:00", 2.25),
    ],
    "bout_a_bout_correction_manuelle": [
        session_fixture("MONTRE", "2030-01-01 23:00", "2030-01-02 03:30", 4.5),
        session_fixture("COMPLEMENT", "2030-01-02 03:30", "2030-01-02 07:30"),
    ],
    "ecart_court_reste_separe": [
        session_fixture("AVANT", "2030-01-01 23:00", "2030-01-02 03:00", 4),
        session_fixture("APRES", "2030-01-02 03:05", "2030-01-02 07:00", 3.9),
    ],
    "chaine_de_chevauchements": [
        session_fixture("G", "2030-01-01 00:00", "2030-01-01 02:00", 2),
        session_fixture("H", "2030-01-01 01:30", "2030-01-01 04:00", 2.5),
        session_fixture("I", "2030-01-01 03:30", "2030-01-01 05:00", 1.5),
    ],
    "journee_corrigee_a_la_main": [
        session_fixture("MATIN", "2030-01-01 07:30", "2030-01-01 09:00", 1.5),
        session_fixture("CORRECTION", "2030-01-01 09:00", "2030-01-01 18:40"),
        session_fixture("APRES_MIDI", "2030-01-01 11:30", "2030-01-01 18:30", 7),
        session_fixture("COMPLEMENT", "2030-01-01 18:30", "2030-01-02 03:40"),
    ],
}

Pour chaque cas : on écrit l'entrée au format du contrat, on calcule les trois sorties avec les fonctions du notebook, et on les écrit à côté.

| Fichier | Contenu |
|---|---|
| `expected_episodes_darkhour.csv` | épisodes d'analyse, règle darkhour (chevauchements seuls) |
| `expected_episodes_nightfall.csv` | épisodes d'analyse, règle Nightfall (+ bout à bout) |
| `expected_displayed.csv` | sessions gardées à l'affichage (doublons ≥ 80 %) |

In [ ]:
def en_texte(instant):
    # Même écriture que l'app : ISO 8601 en UTC, suffixe Z
    return instant.strftime("%Y-%m-%dT%H:%M:%SZ")


def ecrire_fixture(nom, lignes):
    dossier = DOSSIER_FIXTURES / nom
    dossier.mkdir(parents=True, exist_ok=True)
    cas = pd.DataFrame(lignes)

    # ============= Etape 1 : l'entrée, au format du contrat d'export  ===============
    pd.DataFrame({
        "id": cas["id"],
        "start_utc": cas["start_utc"].map(en_texte),
        "end_utc": cas["end_utc"].map(en_texte),
        "start_offset": "+01:00",
        "end_offset": "+01:00",
        "source": "fixture",
        "recording_method": "UNKNOWN",
        "last_modified_utc": cas["last_modified_utc"].map(en_texte),
    }).to_csv(dossier / "sleep_sessions.csv", index=False)

    stades_cas = cas[cas["heures_de_stades"] > 0]
    pd.DataFrame({
        "session_id": stades_cas["id"],
        "start_utc": stades_cas["start_utc"].map(en_texte),
        "end_utc": (stades_cas["start_utc"] + pd.to_timedelta(stades_cas["heures_de_stades"], unit="h")).map(en_texte),
        "stage": "LIGHT",
    }).to_csv(dossier / "sleep_stages.csv", index=False)

    # ============= Etape 2 : les sorties attendues, calculées par le notebook  ===============
    couverture = pd.Series(pd.to_timedelta(cas["heures_de_stades"], unit="h").values, index=cas["id"])
    paires_cas = paires_qui_se_chevauchent(cas)

    for regle, episodes_cas in [
        ("darkhour", episodes_d_analyse(cas, paires_cas)),
        ("nightfall", episodes_nightfall(cas, paires_cas)),
    ]:
        pd.DataFrame({
            "start_utc": episodes_cas["start_utc"].map(en_texte),
            "end_utc": episodes_cas["end_utc"].map(en_texte),
            "member_ids": episodes_cas["membres"],
        }).to_csv(dossier / f"expected_episodes_{regle}.csv", index=False)

    affichees_cas = sessions_affichees(cas, paires_cas, couverture)
    affichees_cas[["id"]].to_csv(dossier / "expected_displayed.csv", index=False)

    return {
        "cas": nom,
        "sessions": len(cas),
        "episodes_darkhour": len(episodes_d_analyse(cas, paires_cas)),
        "episodes_nightfall": len(episodes_nightfall(cas, paires_cas)),
        "affichees": ";".join(affichees_cas["id"]),
    }


bilan = pd.DataFrame([ecrire_fixture(nom, lignes) for nom, lignes in CAS_FIXTURES.items()])
bilan

Les sorties attendues sont vérifiées **à la main** avant d'être figées : une fixture fausse validerait un port Kotlin faux.

In [ ]:
attendu = {
    "session_seule": (1, 1, "S1"),
    "doublon_garde_les_stades": (1, 1, "MONTRE"),
    "doublon_sans_stades_garde_la_plus_longue": (1, 1, "LONGUE"),
    "doublon_identique_garde_la_plus_recente": (1, 1, "RECENTE"),
    "sieste_collee": (1, 1, "SIESTE;APRES"),
    "bout_a_bout_correction_manuelle": (2, 1, "MONTRE;COMPLEMENT"),
    "ecart_court_reste_separe": (2, 2, "AVANT;APRES"),
    "chaine_de_chevauchements": (1, 1, "G;H;I"),
    "journee_corrigee_a_la_main": (2, 1, "MATIN;APRES_MIDI;COMPLEMENT"),
}
for _, ligne in bilan.iterrows():
    assert (ligne["episodes_darkhour"], ligne["episodes_nightfall"], ligne["affichees"]) == attendu[ligne["cas"]], ligne["cas"]

print(f"{len(bilan)} fixtures écrites et vérifiées dans {DOSSIER_FIXTURES}")

# Annexe — Répartition des épisodes par nombre d'heures dormies

Épisodes d'analyse (règle Nightfall), par tranche d'une heure.
La durée va du premier endormissement au dernier réveil de l'épisode : les éveils nocturnes courts sont inclus.

In [ ]:
duree_h = episodes_nf["duree_h"]

resume = pd.Series({
    "Épisodes": len(duree_h),
    "Médiane (h)": round(duree_h.median(), 1),
    "Moyenne (h)": round(duree_h.mean(), 1),
    "Moins de 4 h": f"{(duree_h < 4).mean():.0%}",
    "De 4 à 10 h": f"{((duree_h >= 4) & (duree_h < 10)).mean():.0%}",
    "10 h et plus": f"{(duree_h >= 10).mean():.0%}",
})
resume

In [ ]:
fig = px.histogram(
    episodes_nf,
    x="duree_h",
    color_discrete_sequence=["#0e9eb0"],
    labels={"duree_h": "Heures dormies par épisode"},
)
fig.update_traces(
    xbins=dict(start=0, end=24, size=1),
    marker_line_width=2,
    marker_line_color="white",
    hovertemplate="%{x} h : %{y} épisodes<extra></extra>",
)
fig.add_vline(
    x=duree_h.median(),
    line_dash="dash",
    line_color="#555",
    annotation_text=f"médiane {duree_h.median():.1f} h",
    annotation_position="top right",
)
fig.update_xaxes(dtick=1, range=[0, 22])
fig.update_yaxes(title="Nombre d'épisodes")
fig.update_layout(height=450, bargap=0.05, title="Répartition des épisodes de sommeil par durée")
fig.show()